In [1]:
import os
import shutil
import random
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

DATA_ROOT = "/home/anishma/Desktop/deep_fake_new/FF++"
REAL_SOURCE = os.path.join(DATA_ROOT, "original")
FAKE_SOURCE = os.path.join(DATA_ROOT, "FaceSwap")

OUTPUT_ROOT = "/home/anishma/Desktop/deep_fake_new/FFPP_split(1)"

random.seed(42)

In [2]:
def get_videos(path):
    return sorted([f for f in os.listdir(path) if f.endswith(".mp4")])

real_videos = get_videos(REAL_SOURCE)
fake_videos = get_videos(FAKE_SOURCE)

real_videos = real_videos[:1000]
fake_videos = fake_videos[:1000]

print("Selected Real Videos:", len(real_videos))
print("Selected Fake Videos:", len(fake_videos))

Selected Real Videos: 1000
Selected Fake Videos: 1000


In [3]:
real_train, real_temp = train_test_split(real_videos, test_size=0.30, random_state=42)
real_val, real_test = train_test_split(real_temp, test_size=0.50, random_state=42)

fake_train, fake_temp = train_test_split(fake_videos, test_size=0.30, random_state=42)
fake_val, fake_test = train_test_split(fake_temp, test_size=0.50, random_state=42)

print("Train Real:", len(real_train))
print("Val Real:", len(real_val))
print("Test Real:", len(real_test))

print("Train Fake:", len(fake_train))
print("Val Fake:", len(fake_val))
print("Test Fake:", len(fake_test))

Train Real: 700
Val Real: 150
Test Real: 150
Train Fake: 700
Val Fake: 150
Test Fake: 150


In [4]:
splits = {
    "train": (real_train, fake_train),
    "val": (real_val, fake_val),
    "test": (real_test, fake_test)
}

for split in splits:
    os.makedirs(os.path.join(OUTPUT_ROOT, split, "real"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_ROOT, split, "fake"), exist_ok=True)

print("Folder structure created successfully")

Folder structure created successfully


In [5]:
def move_videos(video_list, source, dest):
    for video in video_list:
        shutil.copy(
            os.path.join(source, video),
            os.path.join(dest, video)
        )

for split, (real_list, fake_list) in splits.items():
    move_videos(real_list, REAL_SOURCE, os.path.join(OUTPUT_ROOT, split, "real"))
    move_videos(fake_list, FAKE_SOURCE, os.path.join(OUTPUT_ROOT, split, "fake"))

print("Videos moved into split folders")

Videos moved into split folders


In [6]:
FACE_ROOT = "/home/anishma/Desktop/deep_fake_new/FFPP_faces(1)"

for split in ["train","val","test"]:
    for cls in ["real","fake"]:
        os.makedirs(
            os.path.join(FACE_ROOT, split, cls),
            exist_ok=True
        )

print("Face dataset folders created")

Face dataset folders created


In [7]:
import dlib
detector = dlib.get_frontal_face_detector()

In [8]:
import cv2
import numpy as np
import os

def extract_faces(video_path, save_dir, target_frames=200):

    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        cap.release()
        return 0

    frame_ids = np.linspace(
        0, total_frames-1,
        target_frames
    ).astype(int)

    saved = 0
    vid_name = os.path.splitext(os.path.basename(video_path))[0]

    for idx in frame_ids:

        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()

        if not ret:
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = detector(gray)

        if len(faces) == 0:
            continue

        face = max(faces, key=lambda r: r.width()*r.height())

        x = max(0, face.left())
        y = max(0, face.top())
        w = face.width()
        h = face.height()

        crop = frame[y:y+h, x:x+w]

        if crop.size == 0:
            continue

        crop = cv2.resize(crop, (224,224))

        filename = f"{vid_name}_{idx}.jpg"

        cv2.imwrite(os.path.join(save_dir, filename), crop)

        saved += 1

    cap.release()
    return saved

In [9]:
VIDEO_ROOT = "/home/anishma/Desktop/deep_fake_new/FFPP_split(1)"
FACE_ROOT  = "/home/anishma/Desktop/deep_fake_new/FFPP_faces(1)"

total_saved = 0

for split in ["train","val","test"]:
    for cls in ["real","fake"]:

        video_folder = os.path.join(VIDEO_ROOT, split, cls)
        save_folder  = os.path.join(FACE_ROOT, split, cls)

        videos = [v for v in os.listdir(video_folder) if v.endswith(".mp4")]

        print(f"\nProcessing {split}/{cls} → {len(videos)} videos")

        for v in tqdm(videos):
            video_path = os.path.join(video_folder, v)

            saved = extract_faces(video_path, save_folder)

            total_saved += saved

print("\nTOTAL FACE IMAGES SAVED:", total_saved)


Processing train/real → 700 videos


100%|███████████████████████████████████████| 700/700 [7:57:06<00:00, 40.89s/it]



Processing train/fake → 700 videos


100%|███████████████████████████████████████| 700/700 [7:42:29<00:00, 39.64s/it]



Processing val/real → 150 videos


...


Processing test/real → 150 videos


100%|███████████████████████████████████████| 150/150 [1:40:15<00:00, 40.10s/it]



Processing test/fake → 150 videos


100%|███████████████████████████████████████| 150/150 [1:37:30<00:00, 39.01s/it]


TOTAL FACE IMAGES SAVED: 399714


In [13]:
import torch

print("GPU Name:", torch.cuda.get_device_name(0))
print("Total VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

GPU Name: Quadro RTX 6000
Total VRAM (GB): 25.19


In [14]:
import torch
import torch.nn as nn
import torch.cuda.amp as amp
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
import os

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: Quadro RTX 6000


In [16]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],
                         [0.5,0.5,0.5])
])

In [17]:
DATA_ROOT = "/home/anishma/Desktop/deep_fake_new/FFPP_faces(1)"

train_dataset = datasets.ImageFolder(
    os.path.join(DATA_ROOT, "train"),
    transform=transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATA_ROOT, "val"),
    transform=transform
)

test_dataset = datasets.ImageFolder(
    os.path.join(DATA_ROOT, "test"),
    transform=transform
)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))

print("Class mapping:", train_dataset.class_to_idx)

Train size: 279754
Val size: 59984
Test size: 59976
Class mapping: {'fake': 0, 'real': 1}


In [18]:
import os
import cv2
import dlib
from tqdm import tqdm

In [19]:
detector = dlib.get_frontal_face_detector()

predictor = dlib.shape_predictor(
    "/home/anishma/Desktop/deep_fake_new/shape_predictor_68_face_landmarks.dat"
)

print("Dlib loaded successfully")

Dlib loaded successfully


In [20]:
FRAME_ROOT = "/home/anishma/Desktop/deep_fake_new/FFPP_faces(1)"
CROP_ROOT  = "/home/anishma/Desktop/deep_fake_new/FFPP_new-faces(1)"

splits = ["train", "val", "test"]
classes = ["real", "fake"]

for split in splits:
    for cls in classes:
        os.makedirs(os.path.join(CROP_ROOT, split, cls), exist_ok=True)

print("Crop folders created")

Crop folders created


In [21]:
def crop_face_from_frame(img_path, save_path):

    img = cv2.imread(img_path)
    if img is None:
        return False

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = detector(gray)

    if len(faces) == 0:
        return False

    # Use largest detected face
    face = max(faces, key=lambda rect: rect.width() * rect.height())

    x = max(0, face.left())
    y = max(0, face.top())
    w = face.width()
    h = face.height()

    crop = img[y:y+h, x:x+w]

    if crop.size == 0:
        return False

    crop = cv2.resize(crop, (224, 224))
    cv2.imwrite(save_path, crop)

    return True

In [22]:
total_saved = 0
total_failed = 0

for split in splits:
    for cls in classes:

        input_folder  = os.path.join(FRAME_ROOT, split, cls)
        output_folder = os.path.join(CROP_ROOT, split, cls)

        images = [f for f in os.listdir(input_folder) if f.endswith(".jpg")]

        print(f"\nProcessing {split}/{cls} - {len(images)} frames")

        for img_name in tqdm(images):

            img_path = os.path.join(input_folder, img_name)
            save_path = os.path.join(output_folder, img_name)

            success = crop_face_from_frame(img_path, save_path)

            if success:
                total_saved += 1
            else:
                total_failed += 1

print("\n==============================")
print("TOTAL CROPPED FACES:", total_saved)
print("TOTAL FAILED:", total_failed)
print("==============================")


Processing train/real - 139927 frames


100%|██████████████████████████████████| 139927/139927 [14:08<00:00, 164.93it/s]



Processing train/fake - 139827 frames


100%|██████████████████████████████████| 139827/139827 [14:48<00:00, 157.32it/s]



Processing val/real - 29994 frames


...


Processing test/real - 29995 frames


100%|████████████████████████████████████| 29995/29995 [03:09<00:00, 158.14it/s]



Processing test/fake - 29981 frames


100%|████████████████████████████████████| 29981/29981 [03:22<00:00, 148.11it/s]


TOTAL CROPPED FACES: 392698
TOTAL FAILED: 7016


In [23]:
def count_images(folder):
    return len([f for f in os.listdir(folder) if f.endswith(".jpg")])

print("\n📊 FINAL CROPPED DATASET SIZE")

total_real = 0
total_fake = 0

for split in splits:
    real_count = count_images(os.path.join(CROP_ROOT, split, "real"))
    fake_count = count_images(os.path.join(CROP_ROOT, split, "fake"))

    total_real += real_count
    total_fake += fake_count

    print(f"\n{split.upper()}")
    print("Real:", real_count)
    print("Fake:", fake_count)
    print("Total:", real_count + fake_count)

print("\nOVERALL")
print("Total Real:", total_real)
print("Total Fake:", total_fake)
print("Grand Total:", total_real + total_fake)


📊 FINAL CROPPED DATASET SIZE

TRAIN
Real: 137407
Fake: 137241
Total: 274648

VAL
Real: 29536
Fake: 29541
Total: 59077

TEST
Real: 29451
Fake: 29522
Total: 58973

OVERALL
Total Real: 196394
Total Fake: 196304
Grand Total: 392698


In [24]:
import sys
print(sys.executable)

/home/anishma/anaconda3/envs/sri/bin/python


In [25]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

DATA_ROOT = "/home/anishma/Desktop/deep_fake_new/FFPP_new-faces(1)"

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

train_dataset = datasets.ImageFolder(
    os.path.join(DATA_ROOT, "train"),
    transform=transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATA_ROOT, "val"),
    transform=transform
)

test_dataset = datasets.ImageFolder(
    os.path.join(DATA_ROOT, "test"),
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 8583
Val batches: 1847
Test batches: 1843


In [26]:
import torch
import torch.nn as nn
import timm
import open_clip

class CNN_CLIP(nn.Module):

    def __init__(self):
        super().__init__()

        # ===== CNN Backbone (same as paper) =====
        self.cnn = timm.create_model(
            "xception",
            pretrained=True,
            num_classes=0
        )

        # CNN output → 2048
        self.projection = nn.Linear(2048, 512)

        # ===== CLIP MODEL =====
        self.clip_model, _, _ = open_clip.create_model_and_transforms(
            'ViT-B-16',
            pretrained='openai'
        )

        # Freeze all CLIP first
        for param in self.clip_model.parameters():
            param.requires_grad = False

        # Unfreeze LAST 4 transformer blocks
        for block in self.clip_model.visual.transformer.resblocks[-4:]:
            for param in block.parameters():
                param.requires_grad = True

        # classifier
        self.fc = nn.Linear(512, 2)

    def forward(self, x):

        # ===== CNN feature =====
        feat = self.cnn.forward_features(x)     # B,2048,7,7
        feat = feat.mean(dim=[2,3])             # GAP → B,2048

        feat = self.projection(feat)            # B,512

        # ===== CLIP expects tokens internally =====
        clip_feat = self.clip_model.visual.forward(x)

        # fuse CNN + CLIP features
        fused = feat + clip_feat

        out = self.fc(fused)

        return out

In [27]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: Quadro RTX 6000


In [28]:
model = CNN_CLIP().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW([
    {"params": model.cnn.parameters(), "lr": 1e-4},
    {"params": model.clip_model.visual.parameters(), "lr": 3e-5},
    {"params": model.fc.parameters(), "lr": 1e-4},
    {"params": model.projection.parameters(), "lr": 1e-4},
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=20
)

/home/anishma/anaconda3/envs/sri/lib/python3.9/site-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(
/home/anishma/anaconda3/envs/sri/lib/python3.9/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


In [29]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [30]:
from tqdm import tqdm

In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import numpy as np

In [32]:
from sklearn.metrics import roc_auc_score

In [33]:
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [34]:
EPOCHS = 20
best_auc = 0

for epoch in range(EPOCHS):

    # ================= TRAIN =================
    model.train()

    train_loss = 0
    train_correct = 0
    total = 0

    for images, labels in tqdm(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        _, preds = torch.max(outputs, 1)

        total += labels.size(0)
        train_correct += (preds == labels).sum().item()

    train_acc = 100 * train_correct / total


    # ================= VALIDATION =================
    model.eval()

    val_correct = 0
    val_total = 0

    all_probs = []
    all_labels = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            probs = torch.softmax(outputs, dim=1)[:,1]

            _, preds = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (preds == labels).sum().item()

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_acc = 100 * val_correct / val_total
    val_auc = roc_auc_score(all_labels, all_probs)

    scheduler.step()

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("Train Loss:", train_loss/len(train_loader))
    print("Train Acc:", train_acc)
    print("Val Acc:", val_acc)
    print("Val AUC:", val_auc)

    # ===== save best =====
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), "best_cnn_clip.pth")
        print("⭐ Best model saved")

100%|███████████████████████████████████████| 8583/8583 [55:47<00:00,  2.56it/s]



Epoch 1/20
Train Loss: 0.021296457036474023
Train Acc: 99.2455797966852
Val Acc: 98.7355485214212
Val AUC: 0.9965231150543364
⭐ Best model saved


100%|███████████████████████████████████████| 8583/8583 [55:13<00:00,  2.59it/s]



Epoch 2/20
Train Loss: 0.007986927006059049
Train Acc: 99.7283795986135
Val Acc: 99.0385429185639
Val AUC: 0.9966385412411192
⭐ Best model saved


100%|███████████████████████████████████████| 8583/8583 [55:13<00:00,  2.59it/s]


...


Epoch 18/20
Train Loss: 2.6503363393658615e-06
Train Acc: 100.0
Val Acc: 99.42786532830036
Val AUC: 0.9973398780733083


100%|███████████████████████████████████████| 8583/8583 [55:01<00:00,  2.60it/s]



Epoch 19/20
Train Loss: 1.6210413774930486e-06
Train Acc: 100.0
Val Acc: 99.4515632141104
Val AUC: 0.9973166901452462


100%|███████████████████████████████████████| 8583/8583 [55:10<00:00,  2.59it/s]



Epoch 20/20
Train Loss: 9.426035324006284e-06
Train Acc: 99.99963589758528
Val Acc: 99.42447991604178
Val AUC: 0.9972518717948352


In [35]:
# ===== Load best model =====
model.load_state_dict(torch.load("best_cnn_clip.pth"))
model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        probs = torch.softmax(outputs, dim=1)

        positive_probs = probs[:,1]

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(positive_probs.cpu().numpy())

test_acc = 100 * (np.array(all_preds) == np.array(all_labels)).mean()
test_auc = roc_auc_score(all_labels, all_probs)

print("\n FINAL TEST ACC:", test_acc)
print(" FINAL TEST AUC:", test_auc)

print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds))


 FINAL TEST ACC: 99.10976209451783
 FINAL TEST AUC: 0.9973907491168046

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.99      0.99     29522
           1       0.99      0.99      0.99     29451

    accuracy                           0.99     58973
   macro avg       0.99      0.99      0.99     58973
weighted avg       0.99      0.99      0.99     58973



In [36]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN_CLIP().to(device)
model.load_state_dict(torch.load("best_cnn_clip.pth"))
model.eval()

print("✅ Model loaded successfully")

/home/anishma/anaconda3/envs/sri/lib/python3.9/site-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(
/home/anishma/anaconda3/envs/sri/lib/python3.9/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


✅ Model loaded successfully


In [37]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [38]:
import dlib
detector = dlib.get_frontal_face_detector()

In [39]:
import cv2
import numpy as np
import torch.nn.functional as F
import os

def predict_video(video_path, max_frames=120):

    print("\n" + "="*60)
    print("Analyzing:", os.path.basename(video_path))
    print("="*60)

    cap = cv2.VideoCapture(video_path)

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        print("❌ Invalid video")
        return

    frame_ids = np.linspace(
        0, total_frames-1,
        max_frames
    ).astype(int)

    fake_probs = []
    real_probs = []

    for idx in frame_ids:

        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()

        if not ret:
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = detector(gray)

        if len(faces) == 0:
            continue

        face = max(faces, key=lambda r: r.width()*r.height())

        x = max(0, face.left())
        y = max(0, face.top())
        w = face.width()
        h = face.height()

        crop = frame[y:y+h, x:x+w]

        if crop.size == 0:
            continue

        crop = cv2.resize(crop,(224,224))
        img = transform(crop).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(img)
            prob = F.softmax(output, dim=1)

            fake_probs.append(prob[0,0].item())
            real_probs.append(prob[0,1].item())

    cap.release()

    if len(fake_probs) == 0:
        print("❌ No face detected")
        return

    avg_fake = np.mean(fake_probs)
    avg_real = np.mean(real_probs)

    if avg_fake > avg_real:
        label = "FAKE"
        conf = avg_fake
    else:
        label = "REAL"
        conf = avg_real

    print("\n🎯 FINAL RESULT:", label)
    print("Confidence:", round(conf*100,2), "%")

    return label, conf

In [41]:
video_path = "/home/anishma/Desktop/deep_fake_new/FF++/FaceSwap/021_312.mp4"

predict_video(video_path)


Analyzing: 021_312.mp4

🎯 FINAL RESULT: FAKE
Confidence: 84.15 %


('FAKE', 0.8415038357698601)

In [42]:
import os

test_folder = "/home/anishma/Desktop/deep_fake_new/FF++/FaceSwap"

count = 0

for v in os.listdir(test_folder):

    if v.endswith(".mp4"):

        path = os.path.join(test_folder, v)

        predict_video(path)

        count += 1

        if count == 10:
            break


Analyzing: 365_307.mp4

🎯 FINAL RESULT: FAKE
Confidence: 59.58 %

Analyzing: 044_945.mp4

🎯 FINAL RESULT: FAKE
Confidence: 100.0 %

Analyzing: 472_511.mp4

🎯 FINAL RESULT: FAKE
Confidence: 99.57 %

Analyzing: 081_087.mp4

🎯 FINAL RESULT: FAKE
Confidence: 97.89 %

Analyzing: 526_436.mp4

🎯 FINAL RESULT: FAKE
Confidence: 99.12 %

Analyzing: 211_177.mp4

🎯 FINAL RESULT: REAL
Confidence: 58.64 %

Analyzing: 753_789.mp4

🎯 FINAL RESULT: FAKE
Confidence: 53.3 %

Analyzing: 034_590.mp4

🎯 FINAL RESULT: FAKE
Confidence: 94.19 %

Analyzing: 760_611.mp4

🎯 FINAL RESULT: FAKE
Confidence: 100.0 %

Analyzing: 531_549.mp4

🎯 FINAL RESULT: FAKE
Confidence: 100.0 %


In [44]:
import os

test_folder = "/home/anishma/Desktop/deep_fake_new/FF++/original"
count = 0

for v in os.listdir(test_folder):

    if v.endswith(".mp4"):

        path = os.path.join(test_folder, v)

        predict_video(path)

        count += 1

        if count == 10:
            break


Analyzing: 213.mp4

🎯 FINAL RESULT: REAL
Confidence: 100.0 %

Analyzing: 541.mp4

🎯 FINAL RESULT: REAL
Confidence: 100.0 %

Analyzing: 712.mp4

🎯 FINAL RESULT: REAL
Confidence: 99.99 %

Analyzing: 867.mp4

🎯 FINAL RESULT: REAL
Confidence: 100.0 %

Analyzing: 203.mp4

🎯 FINAL RESULT: REAL
Confidence: 100.0 %

Analyzing: 023.mp4

🎯 FINAL RESULT: REAL
Confidence: 100.0 %

Analyzing: 987.mp4

🎯 FINAL RESULT: REAL
Confidence: 100.0 %

Analyzing: 205.mp4

🎯 FINAL RESULT: REAL
Confidence: 100.0 %

Analyzing: 598.mp4

🎯 FINAL RESULT: REAL
Confidence: 100.0 %

Analyzing: 723.mp4

🎯 FINAL RESULT: REAL
Confidence: 100.0 %
